# Réplica del paper guía: SegFormer-B0 en ADE20K (ítem 32)

Evalúa el checkpoint publicado de SegFormer-B0 en las 2.000 imágenes de validación de ADE20K, escala única,
con el preprocesamiento de la configuración oficial, y lo compara con el **37,4 % de mIoU** del artículo
según el criterio fijado antes en `config/replica_segformer.json` (δ = 1 punto, IC95 bootstrap por imagen).

- **Qué es:** réplica de **evaluación** (nivel 1). Valida datos, preprocesamiento y métrica; **no** replica el
  entrenamiento. El ajuste corto desde ImageNet (nivel 2) está solo diseñado en el protocolo y este cuaderno no lo ejecuta.
- **Checkpoint:** `Hugging Face nvidia/segformer-b0-finetuned-ade-512-512`, archivo `model.safetensors` con SHA-256 verificado.
- **Datos:** `https://data.csail.mit.edu/places/ADEchallenge/ADEChallengeData2016.zip` (967,382,037 bytes). Términos de uso: Términos de uso de ADE20K en https://ade20k.csail.mit.edu/terms/ (uso académico y no comercial; leerlos antes de descargar). Kit de evaluación CSAILVision/sceneparsing con licencia BSD-3-Clause. El cuaderno exige confirmar la lectura de los términos con una variable antes de descargar.
- **Tiempo estimado en T4 (sin medir aún):** descarga 3-10 min según el servidor del MIT, extracción de validación
  1-2 min, carga del modelo < 1 min, inferencia 3-6 min por variante (2 variantes), bootstrap < 1 min.
  Total previsto 15-30 min. La celda 5 mide 50 imágenes y extrapola antes de la corrida completa.
- **Salidas:** `resumen_replica.json` (veredicto, mIoU, IC95, IoU por clase, desviaciones D1, D2, D4, D5, hashes) y
  `areas_*.npz` (intersección y unión por imagen y clase, para rehacer el bootstrap sin GPU).

Orden: ejecutar las celdas de arriba abajo. Entorno: GPU T4 (Entorno de ejecución > Cambiar tipo > T4).

In [ ]:
# 1. Parámetros. Leer los términos de ADE20K antes de poner True.
ACEPTO_TERMINOS_ADE20K = False      # https://ade20k.csail.mit.edu/terms/
SHA256_ADE_FIJADO = None            # tras la primera descarga, copiar aquí el SHA-256 del registro
GUARDAR_EN_DRIVE = False            # copia resultados a /content/drive/MyDrive/replica_segformer
VARIANTES = ['primaria', 'hf_512']  # primaria decide el veredicto; hf_512 es la desviación D1
RAIZ = '/content/replica'

In [ ]:
# 2. Entorno: GPU y librerías ya incluidas en Colab (no se instala nada).
import importlib, json, platform, subprocess
faltan = [m for m in ['torch', 'transformers', 'huggingface_hub', 'cv2', 'PIL', 'numpy']
          if importlib.util.find_spec(m) is None]
if faltan:
    raise SystemExit(f'Faltan {faltan}. Instalar a mano y anotar la versión en el informe.')
import torch, transformers, cv2, numpy, PIL
print('Python', platform.python_version(), '| torch', torch.__version__, '| transformers', transformers.__version__,
      '| cv2', cv2.__version__, '| numpy', numpy.__version__, '| Pillow', PIL.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NINGUNA (correrá en CPU, mucho más lento)')

In [ ]:
# 3. Escribe el script y el protocolo versionados y verifica su SHA-256.
import hashlib, json, pathlib
ARCHIVOS = {
    'scripts/replica_segformer_ade.py': ("\"\"\"Réplica de evaluación de SegFormer-B0 en ADE20K (ítem 32 del checklist).\n\nSigue config/replica_segformer.json, fijado antes de ejecutar. Corre en Colab\n(lo invoca colab/Replica_SegFormer_ADE20K.ipynb). Las funciones de\npreprocesamiento, acumulación y bootstrap no dependen de transformers y se\nprueban en tests/test_replica_segformer.py con un modelo simulado.\n\nUso en Colab:\n  python replica_segformer_ade.py --ade-root /content/ade/ADEChallengeData2016 \\\n      --salida /content/replica_segformer --variante primaria\n\"\"\"\nimport argparse\nimport hashlib\nimport json\nimport platform\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\nNUM_CLASES = 150\nIGNORAR = 255\nMEDIA_0_255 = np.array([123.675, 116.28, 103.53], dtype=np.float32)\nDE_0_255 = np.array([58.395, 57.12, 57.375], dtype=np.float32)\nMEDIA_IMAGENET = np.array([0.485, 0.456, 0.406], dtype=np.float32)\nDE_IMAGENET = np.array([0.229, 0.224, 0.225], dtype=np.float32)\nHF_MODELO = 'nvidia/segformer-b0-finetuned-ade-512-512'\nHF_REVISION = '489d5cd81a0b59fab9b7ea758d3548ebe99677da'\nHF_SHA256 = '6ae39addd01de6b1b8bde2cf677d43a5cd733424b8d186de3f95d1c51fee23f9'\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, 'rb') as f:\n        for bloque in iter(lambda: f.read(1 << 20), b''):\n            h.update(bloque)\n    return h.hexdigest()\n\n\n# ---------------------------------------------------------------- preprocesamiento\ndef tamano_reescalado(h, w, escala=(2048, 512)):\n    \"\"\"mmcv.rescale_size con keep_ratio: factor min(largo_max/largo, corto_max/corto).\"\"\"\n    largo_max, corto_max = max(escala), min(escala)\n    factor = min(largo_max / max(h, w), corto_max / min(h, w))\n    return int(h * factor + 0.5), int(w * factor + 0.5)\n\n\ndef tamano_alineado(h, w, divisor=32):\n    return int(np.ceil(h / divisor)) * divisor, int(np.ceil(w / divisor)) * divisor\n\n\ndef preprocesar_primario(rgb):\n    \"\"\"AlignedResize de NVlabs/SegFormer: reescala con proporción y alinea a 32.\n\n    Dos redimensiones bilineales con cv2 (como mmcv.imrescale y mmcv.imresize).\n    Devuelve tensor CHW float32 normalizado y el tamaño de entrada.\n    \"\"\"\n    import cv2\n    h, w = rgb.shape[:2]\n    nh, nw = tamano_reescalado(h, w)\n    img = cv2.resize(rgb, (nw, nh), interpolation=cv2.INTER_LINEAR)\n    ah, aw = tamano_alineado(nh, nw)\n    if (ah, aw) != (nh, nw):\n        img = cv2.resize(img, (aw, ah), interpolation=cv2.INTER_LINEAR)\n    img = (img.astype(np.float32) - MEDIA_0_255) / DE_0_255\n    return np.ascontiguousarray(img.transpose(2, 0, 1)), (ah, aw)\n\n\ndef preprocesar_hf(rgb):\n    \"\"\"Variante D1: preprocesador por defecto de la ficha HF (512x512, sin proporción).\n\n    SegformerImageProcessor: resize 512x512 con PIL bilineal (resample=2),\n    reescala a 0-1 y normaliza con media/DE de ImageNet.\n    \"\"\"\n    from PIL import Image\n    img = Image.fromarray(rgb).resize((512, 512), resample=Image.BILINEAR)\n    img = np.asarray(img, dtype=np.float32) / 255.0\n    img = (img - MEDIA_IMAGENET) / DE_IMAGENET\n    return np.ascontiguousarray(img.transpose(2, 0, 1)), (512, 512)\n\n\nPREPROCESOS = {'primaria': preprocesar_primario, 'hf_512': preprocesar_hf}\n\n\ndef reducir_etiqueta_cero(etiqueta):\n    \"\"\"reduce_zero_label de mmseg: 0 -> 255, k -> k-1, y 255 se conserva.\"\"\"\n    e = etiqueta.astype(np.int64)\n    salida = e - 1\n    salida[e == 0] = IGNORAR\n    salida[e == IGNORAR] = IGNORAR\n    return salida\n\n\n# ---------------------------------------------------------------- métrica\ndef interseccion_union(pred, gt, num_clases=NUM_CLASES, ignorar=IGNORAR):\n    \"\"\"Áreas por clase de una imagen, como intersect_and_union de mmseg 0.x.\"\"\"\n    valido = gt != ignorar\n    p = pred[valido].astype(np.int64)\n    g = gt[valido].astype(np.int64)\n    inter = np.bincount(p[p == g], minlength=num_clases)[:num_clases]\n    area_p = np.bincount(p, minlength=num_clases)[:num_clases]\n    area_g = np.bincount(g, minlength=num_clases)[:num_clases]\n    return inter, area_p + area_g - inter, area_g\n\n\ndef miou_acumulado(inter, union):\n    \"\"\"mIoU sobre el conjunto: suma por clase y nanmean de las clases con unión > 0.\"\"\"\n    inter = np.asarray(inter, dtype=np.float64)\n    union = np.asarray(union, dtype=np.float64)\n    if inter.ndim == 2:\n        inter, union = inter.sum(0), union.sum(0)\n    with np.errstate(invalid='ignore', divide='ignore'):\n        iou = np.where(union > 0, inter / np.where(union > 0, union, 1), np.nan)\n    return float(np.nanmean(iou)), iou\n\n\ndef miou_medio_por_imagen(inter, union):\n    \"\"\"Variante D2: media de imágenes del mIoU de cada imagen (no es la métrica del artículo).\"\"\"\n    with np.errstate(invalid='ignore', divide='ignore'):\n        iou = np.where(union > 0, inter / np.where(union > 0, union, 1), np.nan)\n    return float(np.nanmean(np.nanmean(iou, axis=1)))\n\n\ndef bootstrap_miou(inter, union, replicas=2000, semilla=20261120, bloque=200):\n    \"\"\"IC percentil 95 % remuestreando imágenes; recalcula el mIoU acumulado.\"\"\"\n    rng = np.random.default_rng(semilla)\n    n = inter.shape[0]\n    inter = inter.astype(np.float64)\n    union = union.astype(np.float64)\n    valores = np.empty(replicas)\n    hecho = 0\n    while hecho < replicas:\n        k = min(bloque, replicas - hecho)\n        # pesos multinomiales = conteo de veces que entra cada imagen\n        pesos = np.stack([np.bincount(rng.integers(0, n, n), minlength=n) for _ in range(k)])\n        si = pesos @ inter\n        su = pesos @ union\n        with np.errstate(invalid='ignore', divide='ignore'):\n            iou = np.where(su > 0, si / np.where(su > 0, su, 1), np.nan)\n        valores[hecho:hecho + k] = np.nanmean(iou, axis=1)\n        hecho += k\n    return valores\n\n\ndef veredicto(miou_pct, ic_pct, reportado=37.4, delta=1.0):\n    d = miou_pct - reportado\n    ic_d = [ic_pct[0] - reportado, ic_pct[1] - reportado]\n    if abs(d) > delta:\n        v = 'no_replicado'\n    elif -delta <= ic_d[0] and ic_d[1] <= delta:\n        v = 'replicado'\n    else:\n        v = 'compatible'\n    return {'D_puntos': d, 'IC95_D': ic_d, 'veredicto': v,\n            'reportado_dentro_IC95': bool(ic_pct[0] <= reportado <= ic_pct[1]),\n            'delta': delta, 'reportado': reportado}\n\n\n# ---------------------------------------------------------------- evaluación\ndef logits_a_prediccion(logits, tam_entrada, tam_original):\n    \"\"\"Reescala como mmseg: a la entrada y luego al tamaño original; argmax.\"\"\"\n    import torch.nn.functional as F\n    x = F.interpolate(logits, size=tam_entrada, mode='bilinear', align_corners=False)\n    x = F.interpolate(x, size=tam_original, mode='bilinear', align_corners=False)\n    return x.argmax(1)[0].cpu().numpy()\n\n\ndef pares_validacion(ade_root):\n    img_dir = Path(ade_root) / 'images' / 'validation'\n    ann_dir = Path(ade_root) / 'annotations' / 'validation'\n    imagenes = sorted(img_dir.glob('*.jpg'))\n    pares = [(p, ann_dir / (p.stem + '.png')) for p in imagenes]\n    faltan = [str(a) for _, a in pares if not a.exists()]\n    if faltan:\n        raise FileNotFoundError(f'Anotaciones faltantes: {faltan[:3]} (total {len(faltan)})')\n    return pares\n\n\ndef evaluar(modelo_fn, pares, preprocesar, dispositivo, limite=None, avisar=200):\n    \"\"\"modelo_fn(tensor NCHW) -> logits NCHW (150 canales). Devuelve áreas por imagen.\"\"\"\n    import torch\n    from PIL import Image\n    pares = pares[:limite] if limite else pares\n    inter = np.zeros((len(pares), NUM_CLASES), dtype=np.int64)\n    union = np.zeros_like(inter)\n    area_gt = np.zeros_like(inter)\n    nombres = []\n    t0 = time.perf_counter()\n    for i, (ruta_img, ruta_ann) in enumerate(pares):\n        rgb = np.asarray(Image.open(ruta_img).convert('RGB'))\n        gt = reducir_etiqueta_cero(np.asarray(Image.open(ruta_ann)))\n        if gt.shape != rgb.shape[:2]:\n            raise ValueError(f'Tamaños distintos en {ruta_img.name}')\n        x, tam = preprocesar(rgb)\n        with torch.no_grad():\n            logits = modelo_fn(torch.from_numpy(x)[None].to(dispositivo))\n        pred = logits_a_prediccion(logits.float(), tam, gt.shape)\n        inter[i], union[i], area_gt[i] = interseccion_union(pred, gt)\n        nombres.append(ruta_img.stem)\n        if avisar and (i + 1) % avisar == 0:\n            print(f'{i + 1}/{len(pares)} imágenes, {time.perf_counter() - t0:.0f} s', flush=True)\n    return {'nombres': nombres, 'inter': inter, 'union': union, 'area_gt': area_gt,\n            'segundos': time.perf_counter() - t0}\n\n\ndef cargar_modelo_hf(dispositivo):\n    import torch\n    from huggingface_hub import hf_hub_download\n    from transformers import SegformerForSemanticSegmentation\n    ruta = hf_hub_download(HF_MODELO, 'model.safetensors', revision=HF_REVISION)\n    real = sha256(ruta)\n    if real != HF_SHA256:\n        raise RuntimeError(f'SHA-256 del checkpoint distinto: {real}')\n    modelo = SegformerForSemanticSegmentation.from_pretrained(HF_MODELO, revision=HF_REVISION)\n    modelo.eval().to(dispositivo)\n    if modelo.config.num_labels != NUM_CLASES:\n        raise RuntimeError('El checkpoint no tiene 150 clases')\n    parametros = sum(p.numel() for p in modelo.parameters())\n    return (lambda x: modelo(pixel_values=x).logits), {\n        'modelo': HF_MODELO, 'revision': HF_REVISION, 'sha256_safetensors': real,\n        'parametros': parametros, 'id2label': modelo.config.id2label}\n\n\ndef versiones():\n    salida = {'python': platform.python_version()}\n    for m in ['torch', 'transformers', 'huggingface_hub', 'numpy', 'cv2', 'PIL']:\n        try:\n            salida[m] = __import__(m).__version__\n        except Exception as error:  # se registra la ausencia, no se oculta\n            salida[m] = f'no disponible: {error}'\n    try:\n        import torch\n        salida['gpu'] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'\n    except Exception:\n        salida['gpu'] = 'desconocida'\n    return salida\n\n\ndef resumir(areas, protocolo, variante, info_modelo, nombres_clase=None):\n    miou, iou_clase = miou_acumulado(areas['inter'], areas['union'])\n    boot = bootstrap_miou(areas['inter'], areas['union'],\n                          protocolo['incertidumbre']['replicas'], protocolo['incertidumbre']['semilla'])\n    ic = [float(np.nanpercentile(boot, 2.5)), float(np.nanpercentile(boot, 97.5))]\n    crit = protocolo['criterio_a_priori']\n    resultado = {\n        'variante': variante,\n        'es_primaria': variante == 'primaria',\n        'imagenes': len(areas['nombres']),\n        'mIoU_pct': 100 * miou,\n        'IC95_pct': [100 * ic[0], 100 * ic[1]],\n        'bootstrap': {'replicas': len(boot), 'semilla': protocolo['incertidumbre']['semilla'],\n                      'DE_pct': float(100 * np.nanstd(boot))},\n        'mIoU_medio_por_imagen_pct_D2': 100 * miou_medio_por_imagen(areas['inter'], areas['union']),\n        'IoU_por_clase_pct': {\n            (nombres_clase.get(str(k), nombres_clase.get(k, str(k))) if nombres_clase else str(k)):\n                (None if np.isnan(v) else float(100 * v)) for k, v in enumerate(iou_clase)},\n        'segundos_inferencia': areas['segundos'],\n        'modelo': {k: v for k, v in info_modelo.items() if k != 'id2label'},\n        'versiones': versiones(),\n    }\n    resultado['criterio'] = veredicto(resultado['mIoU_pct'], resultado['IC95_pct'],\n                                      protocolo['paper_guia']['valor_reportado_miou'],\n                                      crit['delta_puntos_miou'])\n    if not resultado['es_primaria']:\n        resultado['criterio']['nota'] = 'Variante de análisis de desviación: no decide el veredicto.'\n    return resultado\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument('--ade-root', required=True)\n    parser.add_argument('--salida', required=True)\n    parser.add_argument('--protocolo', default=str(Path(__file__).resolve().parents[1] / 'config/replica_segformer.json'))\n    parser.add_argument('--variante', choices=list(PREPROCESOS), default='primaria')\n    parser.add_argument('--limite', type=int, help='Solo para medir tiempo; invalida el veredicto')\n    args = parser.parse_args()\n    protocolo = json.loads(Path(args.protocolo).read_text(encoding='utf-8'))\n    import torch\n    dispositivo = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n    pares = pares_validacion(args.ade_root)\n    if len(pares) != 2000 and not args.limite:\n        raise RuntimeError(f'Se esperaban 2.000 imágenes de validación, hay {len(pares)}')\n    modelo_fn, info = cargar_modelo_hf(dispositivo)\n    areas = evaluar(modelo_fn, pares, PREPROCESOS[args.variante], dispositivo, args.limite)\n    resumen = resumir(areas, protocolo, args.variante, info, info.get('id2label'))\n    resumen['sha256_protocolo'] = sha256(args.protocolo)\n    if args.limite:\n        resumen['criterio'] = {'veredicto': 'no_aplica', 'nota': f'Corrida parcial de {args.limite} imágenes para medir tiempo'}\n    salida = Path(args.salida)\n    salida.mkdir(parents=True, exist_ok=True)\n    np.savez_compressed(salida / f'areas_{args.variante}.npz', nombres=np.array(areas['nombres']),\n                        inter=areas['inter'], union=areas['union'], area_gt=areas['area_gt'])\n    (salida / f'resumen_{args.variante}.json').write_text(\n        json.dumps(resumen, indent=2, ensure_ascii=False), encoding='utf-8')\n    print(json.dumps({k: resumen[k] for k in ['variante', 'imagenes', 'mIoU_pct', 'IC95_pct', 'criterio']},\n                     indent=2, ensure_ascii=False))\n\n\nif __name__ == '__main__':\n    main()\n", 'f1c545fe6f600abae5a95633444bb2f8fb7ea4e9ae1609b2289c247e13061475'),
    'config/replica_segformer.json': ("{\n  \"schema\": \"replica_paper_guia_v1\",\n  \"estado\": \"Protocolo fijado el 2026-11-20 antes de ejecutar cualquier evaluación en ADE20K. Ningún resultado de la réplica existía al escribirlo. Cambios posteriores solo como enmienda fechada, sin tocar el criterio.\",\n  \"item_checklist\": 32,\n  \"paper_guia\": {\n    \"referencia\": \"Xie, E., Wang, W., Yu, Z., Anandkumar, A., Alvarez, J. M. y Luo, P. (2021). SegFormer: Simple and Efficient Design for Semantic Segmentation with Transformers. Advances in Neural Information Processing Systems 34 (NeurIPS 2021). arXiv:2105.15203.\",\n    \"resultado_clave\": \"SegFormer-B0, ADE20K, mIoU en escala única = 37,4 % (Tabla 2 del artículo; 38,0 % es multiescala con volteo y no se usa).\",\n    \"valor_reportado_miou\": 37.4,\n    \"parametros_reportados_millones\": 3.8\n  },\n  \"niveles\": {\n    \"nivel_1_evaluacion\": \"Evaluar el checkpoint publicado en las condiciones originales. Valida el pipeline de evaluación (datos, preprocesamiento, métrica); NO replica el entrenamiento.\",\n    \"nivel_2_entrenamiento_parcial\": \"Ajuste corto desde los pesos ImageNet-1k del codificador MiT-B0, una semilla. Solo diseño en este protocolo (sección paso_2); no se presenta como ejecutado mientras no exista su resumen JSON.\"\n  },\n  \"condiciones_originales\": {\n    \"dataset\": \"ADE20K, versión SceneParsing/ADEChallengeData2016 (150 clases).\",\n    \"particion\": \"validation: 2.000 imágenes y 2.000 anotaciones (images/validation, annotations/validation). Es la partición con la que el artículo reporta.\",\n    \"metrica\": \"mIoU acumulado sobre el conjunto: por clase, suma de intersecciones / suma de uniones sobre las 2.000 imágenes; media sobre las clases con unión > 0 (nanmean, como mmseg 0.x).\",\n    \"etiquetas\": \"reduce_zero_label: la etiqueta 0 ('otros') pasa a 255 (ignorada) y las 1..150 pasan a 0..149.\",\n    \"configuracion_fuente\": \"https://github.com/NVlabs/SegFormer/blob/65fa8cfa9b52b6ee7e8897a98705abf8570f9e32/local_configs/segformer/B0/segformer.b0.512x512.ade.160k.py y local_configs/_base_/datasets/ade20k_repeat.py (mismo commit)\",\n    \"preprocesamiento_prueba\": {\n      \"lectura\": \"imagen RGB (mmcv lee BGR y convierte con to_rgb=True); anotación PNG leída con Pillow sin cambios\",\n      \"redimension\": \"AlignedResize keep_ratio=True con img_scale=(2048, 512): factor = min(2048/lado_largo, 512/lado_corto); tamaño = int(lado*factor + 0,5); interpolación bilineal (cv2.INTER_LINEAR). Luego segunda redimensión bilineal a múltiplos de 32 por techo (ceil(h/32)*32, ceil(w/32)*32).\",\n      \"normalizacion\": \"media [123.675, 116.28, 103.53], desviación [58.395, 57.12, 57.375] en escala 0-255, RGB, float32\",\n      \"inferencia\": \"test_cfg mode='whole' (imagen completa, sin ventana deslizante), sin volteo, escala única\",\n      \"salida\": \"logits de 1/4 de resolución reescalados bilinealmente (align_corners=False) al tamaño de entrada y luego al tamaño original; argmax sobre 150 clases\"\n    }\n  },\n  \"checkpoint\": {\n    \"fuente_principal\": \"Hugging Face nvidia/segformer-b0-finetuned-ade-512-512, revisión 489d5cd81a0b59fab9b7ea758d3548ebe99677da (consultada el 2026-11-20). La ficha declara que es la conversión del checkpoint publicado en github.com/NVlabs/SegFormer.\",\n    \"archivo\": \"model.safetensors\",\n    \"sha256_lfs\": \"6ae39addd01de6b1b8bde2cf677d43a5cd733424b8d186de3f95d1c51fee23f9\",\n    \"tamano_bytes\": 15036944,\n    \"licencia\": \"NVIDIA Source Code License de NVlabs/SegFormer (uso no comercial de investigación o evaluación); en Hugging Face figura license: other. Se conservan los avisos.\",\n    \"fuente_original_alternativa\": \"segformer.b0.512x512.ade.160k.pth en las carpetas de Google Drive/OneDrive enlazadas desde el README de NVlabs/SegFormer (commit 65fa8cfa). No tiene hash publicado ni descarga directa estable; queda como verificación opcional si se obtiene a mano.\",\n    \"por_que_hugging_face\": \"Menor riesgo de dependencias: transformers ya viene en Colab y carga el modelo en PyTorch puro. La vía oficial depende de mmcv-full 1.x y de la bifurcación mmseg 0.11 de NVlabs, con extensiones compiladas atadas a versiones antiguas de PyTorch y CUDA; instalarlas en el Colab actual obliga a fijar o compilar esas versiones, un riesgo que la vía Hugging Face evita. El riesgo que se acepta a cambio es la fidelidad de la conversión; se controla con el análisis de desviación (D3).\"\n  },\n  \"dataset_descarga\": {\n    \"url\": \"https://data.csail.mit.edu/places/ADEchallenge/ADEChallengeData2016.zip\",\n    \"tamano_bytes_observado\": 967382037,\n    \"etag_observado\": \"\\\"39a91415-5e46c686a7a37\\\"\",\n    \"last_modified_observado\": \"Fri, 22 Jul 2022 22:32:22 GMT\",\n    \"hash_oficial\": null,\n    \"verificacion\": \"No se encontró un hash publicado por la fuente. Se verifica el tamaño contra el observado, se calcula el SHA-256 en la primera descarga y se fija en el resumen; toda ejecución posterior debe dar el mismo SHA-256 o se detiene. Además se exigen 2.000 JPG y 2.000 PNG en validation con nombres emparejados.\",\n    \"licencia_y_terminos\": \"Términos de uso de ADE20K en https://ade20k.csail.mit.edu/terms/ (uso académico y no comercial; leerlos antes de descargar). Kit de evaluación CSAILVision/sceneparsing con licencia BSD-3-Clause. El cuaderno exige confirmar la lectura de los términos con una variable antes de descargar.\",\n    \"uso\": \"Solo evaluación en Colab; no se copia a este equipo ni al repositorio.\"\n  },\n  \"incertidumbre\": {\n    \"unidad\": \"imagen de validación (las 2.000 son escenas distintas)\",\n    \"metodo\": \"bootstrap no paramétrico por imagen: se remuestrean 2.000 imágenes con reemplazo y se recalcula el mIoU acumulado con sus intersecciones y uniones por clase\",\n    \"replicas\": 2000,\n    \"semilla\": 20261120,\n    \"intervalo\": \"percentil 95 %\",\n    \"nota\": \"El IC refleja la variación por muestra de imágenes de validación, no la variación por semilla de entrenamiento (no aplica al nivel 1: un solo checkpoint).\"\n  },\n  \"criterio_a_priori\": {\n    \"delta_puntos_miou\": 1.0,\n    \"justificacion_delta\": \"Un punto de mIoU es una diferencia que no cambia ninguna conclusión del proyecto y tolera diferencias de redimensión y de versión de librería que no se pueden eliminar sin el entorno original (mmcv/mmseg 0.x). Se fija antes de ver resultados.\",\n    \"diferencia\": \"D = mIoU_replica - 37,4 (puntos porcentuales), con IC95 = IC95(mIoU_replica) - 37,4\",\n    \"replicado\": \"IC95 de D completamente dentro de [-1,0; +1,0]\",\n    \"compatible\": \"|D| <= 1,0 pero el IC95 de D sale de [-1,0; +1,0]\",\n    \"no_replicado\": \"|D| > 1,0\",\n    \"secundario\": \"Se informa además si 37,4 cae dentro del IC95 de la réplica, sin que cambie el veredicto.\"\n  },\n  \"analisis_desviacion\": {\n    \"regla\": \"Se ejecuta siempre y se informa completo; ninguna variante reemplaza el resultado primario ni se elige después de verla.\",\n    \"D1_preprocesamiento\": \"Repetir con el preprocesador por defecto de Hugging Face (512x512 sin conservar proporción, normalización ImageNet en escala 0-1). Cuantifica cuánto pesa el redimensionado no documentado en la ficha.\",\n    \"D2_acumulacion\": \"Comparar el mIoU acumulado (primario) con la media por imagen del IoU por clase, para mostrar el efecto de la definición de la métrica.\",\n    \"D3_pesos\": \"Si se obtiene el .pth oficial, comparar sus logits con los de Hugging Face en 20 imágenes fijas (máxima diferencia absoluta). Sin él, se declara como causa no descartada.\",\n    \"D4_versiones\": \"Registrar versiones de torch, transformers, cv2, numpy, Pillow y GPU; el artículo usó mmseg/mmcv con cv2 para redimensionar.\",\n    \"D5_clases\": \"IoU por clase de la réplica; si se dispone del IoU por clase de referencia, listar las 10 clases con mayor diferencia.\",\n    \"si_no_coincide\": \"Se reporta 'no replicado' con D, su IC95 y los resultados D1-D5 como posibles causas. No se cambia el criterio, el delta ni el preprocesamiento primario para forzar la coincidencia.\"\n  },\n  \"paso_2\": {\n    \"estado\": \"Diseño solamente; no ejecutado.\",\n    \"objetivo\": \"Réplica parcial de entrenamiento: verificar que el pipeline de ajuste del paper (codificador ImageNet + cabeza MLP) aprende en ADE20K.\",\n    \"pesos_iniciales\": \"nvidia/mit-b0 en Hugging Face (codificador MiT-B0 preentrenado en ImageNet-1k); fijar revisión y SHA-256 al ejecutar.\",\n    \"receta_del_articulo\": \"AdamW lr 6e-5, cabeza x10, weight decay 0,01, poly power 1 con calentamiento lineal 1.500 iteraciones, recorte 512x512, escala aleatoria 0,5-2,0, volteo, distorsión fotométrica, lote 16, 160.000 iteraciones.\",\n    \"receta_corta\": \"Misma receta con lote 8 y 8.000 iteraciones (5 % del calendario), semilla 11, evaluación con el preprocesamiento primario.\",\n    \"costo_estimado_t4\": \"Estimación sin medir: 1,5 a 2,5 h en T4 con precisión mixta; medir 200 iteraciones antes de comprometer la sesión.\",\n    \"criterio\": \"No se compara con 37,4 % (calendario 20 veces menor). Se informa mIoU con IC95 por imagen y se declara como réplica parcial; si se ejecuta con más de una semilla, se agrega la DE entre semillas.\"\n  }\n}\n", 'fbd963ee4014f429f449ccec23914c141eddb2218130a987efba37aff970e029'),
}
for ruta, (texto, esperado) in ARCHIVOS.items():
    destino = pathlib.Path(RAIZ) / ruta
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_bytes(texto.encode('utf-8'))
    real = hashlib.sha256(destino.read_bytes()).hexdigest()
    assert real == esperado, f'SHA-256 distinto en {ruta}'
    print(ruta, real[:16])
import sys; sys.path.insert(0, f'{RAIZ}/scripts')
import replica_segformer_ade as R
PROTOCOLO = json.loads(pathlib.Path(RAIZ, 'config/replica_segformer.json').read_text(encoding='utf-8'))

In [ ]:
# 4. ADE20K desde la fuente oficial: tamaño, SHA-256, conteo y registro de licencia.
import os, time, urllib.request, zipfile, datetime
assert ACEPTO_TERMINOS_ADE20K, 'Leer los términos de ADE20K y poner ACEPTO_TERMINOS_ADE20K = True'
URL = 'https://data.csail.mit.edu/places/ADEchallenge/ADEChallengeData2016.zip'
TAMANO = 967382037
ZIP = '/content/ade/ADEChallengeData2016.zip'
os.makedirs('/content/ade', exist_ok=True)
if not os.path.exists(ZIP) or os.path.getsize(ZIP) != TAMANO:
    t0 = time.time()
    with urllib.request.urlopen(URL) as r, open(ZIP, 'wb') as f:
        cabeceras = dict(r.headers)
        while bloque := r.read(1 << 22):
            f.write(bloque)
    print(f'Descarga: {time.time() - t0:.0f} s')
else:
    cabeceras = {}
assert os.path.getsize(ZIP) == TAMANO, f'Tamaño distinto: {os.path.getsize(ZIP)}'
SHA_ADE = R.sha256(ZIP)
print('SHA-256 ADEChallengeData2016.zip:', SHA_ADE)
if SHA256_ADE_FIJADO:
    assert SHA_ADE == SHA256_ADE_FIJADO, 'El zip no coincide con el fijado en una corrida anterior'
t0 = time.time()
with zipfile.ZipFile(ZIP) as z:
    miembros = [m for m in z.namelist() if '/validation/' in m and not m.endswith('/')]
    z.extractall('/content/ade', members=miembros)
ADE_ROOT = '/content/ade/ADEChallengeData2016'
PARES = R.pares_validacion(ADE_ROOT)
assert len(PARES) == 2000, len(PARES)
print(f'Extracción de validación: {time.time() - t0:.0f} s, {len(PARES)} pares imagen/anotación')
REGISTRO_DATOS = {'url': URL, 'tamano_bytes': TAMANO, 'sha256': SHA_ADE,
    'sha256_era_fijado': bool(SHA256_ADE_FIJADO), 'etag': cabeceras.get('ETag'),
    'last_modified': cabeceras.get('Last-Modified'), 'imagenes_validacion': len(PARES),
    'terminos': 'https://ade20k.csail.mit.edu/terms/', 'terminos_aceptados_por_quien_ejecuta': True,
    'licencia_kit': 'CSAILVision/sceneparsing BSD-3-Clause',
    'fecha_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z'}

In [ ]:
# 5. Modelo con revisión y SHA-256 fijados; medición de 50 imágenes y extrapolación.
DISPOSITIVO = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODELO_FN, INFO = R.cargar_modelo_hf(DISPOSITIVO)
print(INFO['modelo'], INFO['revision'][:12], 'parámetros:', f"{INFO['parametros']:,}")
prueba = R.evaluar(MODELO_FN, PARES, R.preprocesar_primario, DISPOSITIVO, limite=50, avisar=0)
por_imagen = prueba['segundos'] / 50
print(f'{por_imagen*1000:.0f} ms por imagen -> {por_imagen*2000/60:.1f} min por variante (2.000 imágenes)')

In [ ]:
# 6. Evaluación completa de cada variante. La primaria decide el veredicto.
import numpy as np
SALIDA = pathlib.Path('/content/replica_segformer'); SALIDA.mkdir(exist_ok=True)
RESULTADOS = {}
for variante in VARIANTES:
    areas = R.evaluar(MODELO_FN, PARES, R.PREPROCESOS[variante], DISPOSITIVO)
    np.savez_compressed(SALIDA / f'areas_{variante}.npz', nombres=np.array(areas['nombres']),
                        inter=areas['inter'], union=areas['union'], area_gt=areas['area_gt'])
    RESULTADOS[variante] = R.resumir(areas, PROTOCOLO, variante, INFO, INFO['id2label'])
    r = RESULTADOS[variante]
    print(f"{variante}: mIoU {r['mIoU_pct']:.2f} % IC95 [{r['IC95_pct'][0]:.2f}; {r['IC95_pct'][1]:.2f}] "
          f"| media por imagen (D2) {r['mIoU_medio_por_imagen_pct_D2']:.2f} % | {areas['segundos']/60:.1f} min")

In [ ]:
# 7. Resumen único: veredicto a priori y análisis de la desviación (D1, D2, D4, D5).
p = RESULTADOS['primaria']
resumen = {
    'item_checklist': 32, 'nivel': 'nivel_1_evaluacion (no replica el entrenamiento)',
    'sha256_protocolo': hashlib.sha256(pathlib.Path(RAIZ, 'config/replica_segformer.json').read_bytes()).hexdigest(),
    'sha256_script': hashlib.sha256(pathlib.Path(RAIZ, 'scripts/replica_segformer_ade.py').read_bytes()).hexdigest(),
    'dataset': REGISTRO_DATOS, 'modelo': p['modelo'], 'versiones_D4': p['versiones'],
    'reportado_articulo_pct': PROTOCOLO['paper_guia']['valor_reportado_miou'],
    'primaria': {k: v for k, v in p.items() if k != 'IoU_por_clase_pct'},
    'veredicto': p['criterio']['veredicto'],
    'D1_preprocesamiento_hf_512': ({'mIoU_pct': RESULTADOS['hf_512']['mIoU_pct'],
                                    'IC95_pct': RESULTADOS['hf_512']['IC95_pct'],
                                    'diferencia_con_primaria': RESULTADOS['hf_512']['mIoU_pct'] - p['mIoU_pct']}
                                   if 'hf_512' in RESULTADOS else 'no ejecutada'),
    'D2_media_por_imagen_pct': p['mIoU_medio_por_imagen_pct_D2'],
    'D3_pesos': 'No evaluada: sin el .pth oficial no se descarta una diferencia de conversión.',
    'D5_IoU_por_clase_pct': p['IoU_por_clase_pct'],
}
(SALIDA / 'resumen_replica.json').write_text(json.dumps(resumen, indent=2, ensure_ascii=False), encoding='utf-8')
clases = sorted(((v, k) for k, v in p['IoU_por_clase_pct'].items() if v is not None))
print('Veredicto:', resumen['veredicto'], '| D =', round(p['criterio']['D_puntos'], 2), 'puntos, IC95',
      [round(x, 2) for x in p['criterio']['IC95_D']])
print('37,4 dentro del IC95:', p['criterio']['reportado_dentro_IC95'])
print('Peores 5 clases:', [(k, round(v, 1)) for v, k in clases[:5]])

In [ ]:
# 8. Guardar: Drive opcional y zip para descargar (subirlo al repo en resultados/replica_segformer/).
import shutil
if GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copytree(SALIDA, '/content/drive/MyDrive/replica_segformer', dirs_exist_ok=True)
shutil.make_archive('/content/replica_segformer', 'zip', SALIDA)
print('Listo: /content/replica_segformer.zip')

## Cómo informar

- Citar el veredicto de `resumen_replica.json` tal como sale (`replicado`, `compatible` o `no_replicado`), con D y su IC95.
- Si no es `replicado`, presentar D1, D2, D4 y D5 como posibles causas; D3 queda como causa no descartada.
- No cambiar el δ, el preprocesamiento primario ni la métrica después de ver el número.
- Esto evalúa un checkpoint: no dice que el entrenamiento del artículo se haya reproducido.